# Baseline SVM Nível 2 (área responsável)

Treinamento e avaliação do baseline **TF-IDF + LinearSVC** para classificação de tickets ITIL no Nível 2 (área técnica responsável). Espelha a metodologia de avaliação do notebook BERT (`bert_nivel2_training.ipynb`) para permitir comparação direta.

- **Input:** `dataset_nivel_2.csv` com colunas `descricao_limpa` e `area_responsavel`
- **Modelo:** `TfidfVectorizer(1-2gram)` + `CalibratedClassifierCV(LinearSVC(class_weight="balanced"))` idêntico ao baseline de Nível 1
- **Classes (5):** Aplicações, Servidor, Rede, Segurança, BD
- **Avaliação:** Stratified 5-Fold com predições out-of-fold (mesma do BERT)
- **Output:** modelo em `backend/models/baseline/svm_nivel2.pkl`

> **Por que o SVM usa `descricao_limpa` e o BERT usa `descricao` (cru)?** Cada modelo recebe a representação para a qual foi projetado: TF-IDF (bag-of-words) se beneficia da remoção de acentos/stopwords; o BERTimbau (*cased*, pré-treinado em texto natural) se beneficia do texto cru. Comparar cada um no seu input ótimo é a prática correta.

In [ ]:
from pathlib import Path

import pandas as pd

# Roda tanto localmente (qualquer cwd dentro do repo) quanto no Colab (upload).
CANDIDATOS = [
    Path("data/dataset_nivel_2.csv"),
    Path("../data/dataset_nivel_2.csv"),
    Path(__file__).resolve().parent.parent / "data/dataset_nivel_2.csv"
    if "__file__" in dir()
    else Path("data/dataset_nivel_2.csv"),
]
local = next((c for c in CANDIDATOS if c.exists()), None)

if local is not None:
    df = pd.read_csv(local)
    print(f"Lido localmente: {local}")
else:
    from google.colab import files

    uploaded = files.upload()  # selecione o dataset_nivel_2.csv
    df = pd.read_csv(list(uploaded.keys())[0])

# Compatibilidade: aceita tanto 'area_responsavel' quanto o antigo 'nivel2'
if "area_responsavel" not in df.columns and "nivel2" in df.columns:
    df = df.rename(columns={"nivel2": "area_responsavel"})

print(f"Linhas: {len(df)}")
print(df["area_responsavel"].value_counts())

In [ ]:
import numpy as np
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

RANDOM_SEED = 42
TARGET_COL = "area_responsavel"
TEXT_COL = (
    "descricao_limpa"  # TF-IDF se beneficia do texto limpo (ao contrario do BERT)
)

df = df.dropna(subset=[TARGET_COL, TEXT_COL]).reset_index(drop=True)
CLASSES = sorted(df[TARGET_COL].unique())
print(f"Classes ({len(CLASSES)}): {CLASSES}")


def build_pipeline():
    """TF-IDF + LinearSVC calibrado identico ao baseline de Nivel 1."""
    return Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    ngram_range=(1, 2),
                    min_df=2,
                    max_df=0.95,
                    sublinear_tf=True,
                ),
            ),
            (
                "clf",
                CalibratedClassifierCV(
                    LinearSVC(max_iter=2000, class_weight="balanced"),
                    cv=3,
                ),
            ),
        ]
    )

Classes (5): ['Aplicações', 'BD', 'Rede', 'Segurança', 'Servidor']


## 1. Split único (visão rápida)

Mesmo split do BERT (20% estratificado, seed 42). Serve só de referência rápida, o número confiável é o do K-Fold abaixo.

In [3]:
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

X = df[TEXT_COL].fillna("").tolist()
y = df[TARGET_COL].tolist()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

pipe = build_pipeline()
pipe.fit(X_tr, y_tr)
y_pred = pipe.predict(X_te)

print(classification_report(y_te, y_pred, labels=CLASSES, zero_division=0))

              precision    recall  f1-score   support

  Aplicações       0.68      0.88      0.77        17
          BD       1.00      0.33      0.50         3
        Rede       0.75      0.50      0.60         6
   Segurança       0.50      0.60      0.55         5
    Servidor       0.50      0.33      0.40         6

    accuracy                           0.65        37
   macro avg       0.69      0.53      0.56        37
weighted avg       0.66      0.65      0.63        37



## 2. Avaliação robusta Stratified 5-Fold (out-of-fold)

Idêntica ao bloco K-Fold do notebook BERT: cada ticket é predito uma vez (no fold de teste), gerando predições OOF para os 185 tickets. O report agregado usa o **support real** (BD = 13) e reportamos **média ± desvio** do F1 macro.

In [ ]:
import json

from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold

N_FOLDS = 5

X = np.array(df[TEXT_COL].fillna("").tolist(), dtype=object)
y = np.array(df[TARGET_COL].tolist(), dtype=object)

oof_pred = np.empty(len(df), dtype=object)
fold_f1 = []

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
    pipe = build_pipeline()
    pipe.fit(X[tr_idx].tolist(), y[tr_idx].tolist())
    preds = pipe.predict(X[te_idx].tolist())
    oof_pred[te_idx] = preds

    f1 = f1_score(y[te_idx], preds, average="macro", zero_division=0)
    fold_f1.append(f1)
    print(f"Fold {fold}: F1 macro = {f1:.3f}")

print(f"\nF1 macro por fold: {[round(f, 3) for f in fold_f1]}")
print(f"F1 macro CV (5-fold): {np.mean(fold_f1):.3f} +/- {np.std(fold_f1):.3f}")

print("\n=== Classification report agregado (OOF todos os 185 tickets) ===")
print(
    classification_report(
        y, oof_pred, labels=CLASSES, target_names=CLASSES, zero_division=0
    )
)

print("=== Matriz de confusao (OOF) ===")
print("Classes:", CLASSES)
print(confusion_matrix(y, oof_pred, labels=CLASSES))

cv_results = {
    "model": "tfidf+linearsvc",
    "n_folds": N_FOLDS,
    "text_col": TEXT_COL,
    "f1_macro_per_fold": [round(f, 4) for f in fold_f1],
    "f1_macro_mean": round(float(np.mean(fold_f1)), 4),
    "f1_macro_std": round(float(np.std(fold_f1)), 4),
    "oof_classification_report": classification_report(
        y,
        oof_pred,
        labels=CLASSES,
        target_names=CLASSES,
        output_dict=True,
        zero_division=0,
    ),
}
with open("svm_nivel2_cv_results.json", "w") as f:
    json.dump(cv_results, f, indent=2, ensure_ascii=False)
print("\nResultados do CV salvos em svm_nivel2_cv_results.json")

Fold 1: F1 macro = 0.604


Fold 2: F1 macro = 0.824
Fold 3: F1 macro = 0.679


Fold 4: F1 macro = 0.524
Fold 5: F1 macro = 0.625

F1 macro por fold: [np.float64(0.604), np.float64(0.824), np.float64(0.679), np.float64(0.524), np.float64(0.625)]
F1 macro CV (5-fold): 0.651 +/- 0.100

=== Classification report agregado (OOF — todos os 185 tickets) ===
              precision    recall  f1-score   support

  Aplicações       0.75      0.90      0.81        86
          BD       0.50      0.46      0.48        13
        Rede       0.79      0.52      0.62        29
   Segurança       0.59      0.57      0.58        28
    Servidor       0.79      0.66      0.72        29

    accuracy                           0.72       185
   macro avg       0.68      0.62      0.64       185
weighted avg       0.72      0.72      0.71       185

=== Matriz de confusao (OOF) ===
Classes: ['Aplicações', 'BD', 'Rede', 'Segurança', 'Servidor']
[[77  3  1  5  0]
 [ 3  6  0  2  2]
 [ 8  1 15  3  2]
 [ 9  0  2 16  1]
 [ 6  2  1  1 19]]

Resultados do CV salvos em svm_nivel2_cv_results.j

## 3. Modelo final (deploy)

Treina o pipeline em **todos os 185 tickets** (após o CV validar a abordagem) e salva em `backend/models/baseline/svm_nivel2.pkl`.

In [ ]:
import joblib

REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend").exists()),
    Path.cwd(),
)
MODEL_PATH = REPO / "backend/models/baseline/svm_nivel2.pkl"
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

final_pipe = build_pipeline()
final_pipe.fit(df[TEXT_COL].fillna("").tolist(), df[TARGET_COL].tolist())
joblib.dump(final_pipe, MODEL_PATH)
print(f"Modelo salvo em: {MODEL_PATH}")

# No Colab, baixa o arquivo
try:
    from google.colab import files

    files.download(str(MODEL_PATH))
except Exception:
    pass

## Próximos passos

1. Compare o **F1 macro CV** do SVM com o do BERT (`0.5276 ± 0.1252`, ver `bert_nivel2_cv_results.json` e `notebooks/comparacao_svm_bert.ipynb`). O SVM supera o BERT, achado central: em regime de poucos dados, o baseline clássico é melhor que o transformer, não apenas competitivo.
2. Compare as **matrizes de confusão OOF** dos dois modelos, provavelmente ambos confundem Rede/Segurança/Aplicações, o que reforça a sobreposição semântica (e não uma fraqueza específica de um modelo).
3. A augmentation focada (BD, Rede) beneficia os dois; ao adicioná-la, migrar o CV para `StratifiedGroupKFold` (grupo = ticket original) para evitar vazamento.
